# Classification → Regression (2 ขั้นตอน, ใช้ sklearn ได้)

**Flow:** `features` → [Stage 1] ทำนาย **class** → เอาผลทำนาย class (และ probability) มาเป็น feature เพิ่ม → [Stage 2] ทำนาย **ค่าต่อเนื่อง (y)**

| Stage | วัดผลด้วย |
|---|---|
| 1 Classification | Accuracy, F1 (macro/weighted), confusion matrix |
| 2 Regression | R², RMSE, MAE |

**ลำดับการรัน:** 0 โหลด → 1 config+สำรวจ → 2 data prep → 3 Stage 1 → 4 สร้าง feature ให้ Stage 2 → 5 Stage 2 → 6 ทำนายแบบ end-to-end → 7 ส่วนเสริม

**จุดที่ต้องระวังที่สุด (ไฟล์นี้จัดการให้แล้ว):**
- ตอน train Stage 2 ห้ามใช้ class ที่ model ทำนายบน train ตรงๆ (มันแม่นเกินจริง) ไฟล์นี้ใช้ **out-of-fold prediction** แทน เพื่อให้ feature ตอน train หน้าตาเหมือนตอน test
- ทุกอย่างเรียนรู้จาก train เท่านั้น (impute, scale, encode อยู่ใน Pipeline)
- มีตาราง **ablation** เทียบ "ไม่ใช้ class / ใช้ class ที่ทำนาย / ใช้ class จริง" ให้เห็นว่า Stage 1 ช่วย Stage 2 จริงไหม

## 0. โหลดข้อมูล

In [ ]:
import pandas as pd

# แบบ 1: ไฟล์ csv อยู่ใน Colab แล้ว
df = pd.read_csv('data.csv')

# แบบ 2: ต้องอัปโหลดเอง
# from google.colab import files
# files.upload()
# df = pd.read_csv('ชื่อไฟล์.csv')

# แบบ 3: อาจารย์ให้ dataframe มาแล้ว (เช่นชื่อ data) -> df = data.copy()

df.head()

## 1. Config + สำรวจข้อมูล

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ============ CONFIG (แก้ที่นี่ที่เดียว) ============
CLASS_COL = 'ชื่อคอลัมน์class'    # คอลัมน์ class (ที่ต้อง classify)
REG_COL   = 'ชื่อคอลัมน์y'      # คอลัมน์ y ต่อเนื่อง (ที่ต้อง regress)
DROP_COLS = []               # คอลัมน์ที่จะทิ้ง เช่น ['id', 'name', 'date']

TEST_SIZE      = 0.2
SEED           = 42
N_SPLITS       = 5           # จำนวน fold ของ CV

MAX_NULL_COL   = 0.50        # คอลัมน์ null เกินนี้ -> ทิ้ง
ADD_INDICATOR  = 0.05        # คอลัมน์ null เกินนี้ -> เพิ่ม flag _missing

CLS_METRIC     = 'f1_macro'  # ตัวที่ใช้จัดอันดับ Stage 1: 'accuracy' | 'f1_macro' | 'f1_weighted'
CLASS_WEIGHT   = None        # None | 'balanced' (ใช้ถ้า class ไม่สมดุล)

USE_LOG_TARGET   = False     # True = regress บน log1p(y) (ผลวัดยังเป็นหน่วยเดิม)
ADD_PROBA        = True      # ส่ง probability ของแต่ละ class ไปเป็น feature ใน Stage 2
ADD_CLASS_ONEHOT = True      # ส่ง class ที่ทำนาย (one-hot) ไปเป็น feature ใน Stage 2
# ====================================================

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('shape:', df.shape)
df.info()

# --- null ---
nr = pd.DataFrame({'n_null': df.isna().sum(), 'pct_null': (df.isna().mean() * 100).round(2)})
display(nr[nr.n_null > 0].sort_values('pct_null', ascending=False))

# --- class ---
vc = df[CLASS_COL].value_counts()                      # ไม่นับแถวที่ class เป็น null
display(pd.DataFrame({'count': vc, 'pct': (vc / vc.sum() * 100).round(1)}))
print('แถวที่ class เป็น null (จะถูกทิ้ง):', int(df[CLASS_COL].isna().sum()))
print(f'Baseline accuracy (ทายเป็น class ใหญ่สุดตลอด) = {vc.max() / vc.sum():.3f}  <- model ต้องชนะค่านี้')
if vc.min() / vc.max() < 0.3:
    print('!! class ไม่สมดุล -> ใช้ F1 เป็นหลัก และลอง CLASS_WEIGHT = "balanced"')

# --- y ตาม class: Stage 1 จะช่วย Stage 2 ได้ก็ต่อเมื่อ y ต่างกันตาม class ---
display(df.groupby(CLASS_COL)[REG_COL].agg(['count', 'mean', 'std', 'min', 'median', 'max']).round(3))
tmp = df[[CLASS_COL, REG_COL]].dropna()
g = tmp[REG_COL].mean()
eta2 = tmp.groupby(CLASS_COL)[REG_COL].apply(lambda s: len(s) * (s.mean() - g) ** 2).sum() / ((tmp[REG_COL] - g) ** 2).sum()
print(f'eta^2 (สัดส่วนความแปรปรวนของ y ที่ class อธิบายได้) = {eta2:.3f}')
print('  ใกล้ 0 = class แทบไม่ช่วยทำนาย y | สูง (>0.3) = class ช่วยมาก คุ้มที่จะทำ Stage 1 ให้แม่น')
print('skew ของ y =', round(tmp[REG_COL].skew(), 2), '(เกิน ±1 ลอง USE_LOG_TARGET = True)')

tmp.boxplot(column=REG_COL, by=CLASS_COL, figsize=(6, 4)); plt.suptitle(''); plt.title('y แยกตาม class'); plt.show()

## 2. Data prep + split (ต้องรัน)

- ทิ้งแถวที่ไม่มี class หรือ y, ทิ้งคอลัมน์ที่ null เกินเกณฑ์, เพิ่ม flag `_missing`
- impute / scale / one-hot อยู่ใน **Pipeline** (`make_prep`) → fit เฉพาะ train และใช้กับข้อมูลใหม่ได้อัตโนมัติ
- split แบบ **stratified** ตาม class เพื่อให้สัดส่วน class ใน train/test ใกล้กัน

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

data = df.drop_duplicates().copy()
data = data.dropna(subset=[CLASS_COL, REG_COL])
data = data.drop(columns=[c for c in DROP_COLS if c in data.columns])

feat = [c for c in data.columns if c not in (CLASS_COL, REG_COL)]
ratio = data[feat].isna().mean()
too_null = ratio[ratio > MAX_NULL_COL].index.tolist()
feat = [c for c in feat if c not in too_null]
print('ทิ้งคอลัมน์ null เยอะ:', too_null)

ind_cols = [c for c in feat if data[c].isna().mean() > ADD_INDICATOR]
print('เพิ่ม flag _missing:', ind_cols)

raw_feat = feat.copy()                                            # คอลัมน์ที่ข้อมูลใหม่ต้องมี
cat_cols = data[feat].select_dtypes(exclude='number').columns.tolist()
num_cols = [c for c in feat if c not in cat_cols] + [c + '_missing' for c in ind_cols]
print('numeric:', len(num_cols), '| categorical:', cat_cols)

def clean_X(d_):
    """ทำความสะอาดแบบเดียวกันกับข้อมูลใหม่ (ใช้ซ้ำตอน predict)"""
    X_ = d_[raw_feat].copy()
    for c in ind_cols:
        X_[c + '_missing'] = X_[c].isna().astype(int)
    for c in cat_cols:
        X_[c] = X_[c].astype(object).fillna('missing').astype(str)
    return X_

def make_prep():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ])

X_all = clean_X(data)
le = LabelEncoder()
yc_all = le.fit_transform(data[CLASS_COL])
class_names = [str(c) for c in le.classes_]
yr_all = data[REG_COL].to_numpy(float)
K = len(class_names)
print('classes:', dict(enumerate(class_names)))

try:
    idx_tr, idx_te = train_test_split(np.arange(len(X_all)), test_size=TEST_SIZE, random_state=SEED, stratify=yc_all)
except ValueError:
    print('!! stratify ไม่ได้ (บาง class น้อยเกิน) -> split แบบสุ่มธรรมดา')
    idx_tr, idx_te = train_test_split(np.arange(len(X_all)), test_size=TEST_SIZE, random_state=SEED)

X_train, X_test = X_all.iloc[idx_tr], X_all.iloc[idx_te]
yc_train, yc_test = yc_all[idx_tr], yc_all[idx_te]
yr_train, yr_test = yr_all[idx_tr], yr_all[idx_te]
print('train:', X_train.shape, ' test:', X_test.shape)

## 3. Stage 1: Classification

### 3.1 เทียบโมเดล (จัดอันดับด้วย CV บน train, ดู test ประกอบ)
มี Baseline (ทายเป็น class ใหญ่สุด) ไว้เทียบ ถ้า model ชนะ baseline ไม่มาก แปลว่า feature แทบไม่บอกอะไร

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier

K_CV = max(2, min(N_SPLITS, int(np.bincount(yc_train).min())))
cv_cls = StratifiedKFold(K_CV, shuffle=True, random_state=SEED)
cw = CLASS_WEIGHT

CLS_MODELS = {
    'Logistic':     LogisticRegression(max_iter=2000, class_weight=cw),
    'KNN':          KNeighborsClassifier(n_neighbors=7),
    'SVC':          SVC(probability=True, class_weight=cw, random_state=SEED),   # probability=True เพื่อให้ได้ proba (ช้าลงเล็กน้อย)
    'NaiveBayes':   GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(max_depth=6, class_weight=cw, random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight=cw, random_state=SEED, n_jobs=-1),
    'GradBoost':    GradientBoostingClassifier(random_state=SEED),
}

def cls_pipe(est):
    return Pipeline([('prep', make_prep()), ('clf', est)])

scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro', 'f1_weighted': 'f1_weighted'}
rows = []
for name, est in {'Baseline(majority)': DummyClassifier(strategy='most_frequent'), **CLS_MODELS}.items():
    t0 = time.time()
    pipe = cls_pipe(est)
    cvr = cross_validate(pipe, X_train, yc_train, cv=cv_cls, scoring=scoring, n_jobs=-1)
    pipe.fit(X_train, yc_train)
    p = pipe.predict(X_test)
    rows.append({'model': name,
                 'CV_accuracy': cvr['test_accuracy'].mean(), 'CV_f1_macro': cvr['test_f1_macro'].mean(),
                 'CV_f1_weighted': cvr['test_f1_weighted'].mean(),
                 'Test_acc': accuracy_score(yc_test, p),
                 'Test_f1_macro': f1_score(yc_test, p, average='macro'),
                 'Test_f1_weighted': f1_score(yc_test, p, average='weighted')})
    print(f'{name:<20} CV_{CLS_METRIC}={rows[-1]["CV_" + CLS_METRIC]:.4f}  [{time.time() - t0:.1f}s]')

cls_df = pd.DataFrame(rows).sort_values('CV_' + CLS_METRIC, ascending=False).reset_index(drop=True)
cls_df.insert(0, 'rank', cls_df.index + 1)
display(cls_df.round(4))

### 3.2 จูน hyperparameter ของโมเดลที่ชนะ + fit ตัวสุดท้าย (`clf_final`)
เลือกด้วย CV บน train ไม่ใช้ test ถ้าไม่อยากจูน ตั้ง `TUNE_CLS = False`

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

TUNE_CLS = True
CLS_PICK = None        # None = ตัวอันดับ 1 (ไม่นับ baseline) | หรือระบุชื่อ เช่น 'RandomForest'

GRIDS_CLS = {
    'Logistic':     {'clf__C': [0.01, 0.1, 1, 10, 100]},
    'KNN':          {'clf__n_neighbors': [3, 5, 7, 11, 15, 25], 'clf__weights': ['uniform', 'distance']},
    'SVC':          {'clf__C': [0.1, 1, 10, 100], 'clf__gamma': ['scale', 0.01, 0.1]},
    'NaiveBayes':   {'clf__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]},
    'DecisionTree': {'clf__max_depth': [3, 5, 8, 12, None], 'clf__min_samples_leaf': [1, 5, 10, 20]},
    'RandomForest': {'clf__max_depth': [None, 5, 10, 20], 'clf__min_samples_leaf': [1, 2, 5], 'clf__max_features': ['sqrt', 0.5]},
    'GradBoost':    {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1], 'clf__max_depth': [2, 3, 4]},
}

cls_pick = CLS_PICK or cls_df[cls_df.model != 'Baseline(majority)'].iloc[0]['model']
pipe = cls_pipe(clone(CLS_MODELS[cls_pick]))
if TUNE_CLS and cls_pick in GRIDS_CLS:
    gs = GridSearchCV(pipe, GRIDS_CLS[cls_pick], cv=cv_cls, scoring=CLS_METRIC, n_jobs=-1)
    gs.fit(X_train, yc_train)
    clf_final = gs.best_estimator_
    print('โมเดล:', cls_pick, '| best params:', gs.best_params_, f'| CV {CLS_METRIC}={gs.best_score_:.4f}')
    # เช็คว่าค่าที่ชนะอยู่ขอบ grid ไหม
    for k_, vals in GRIDS_CLS[cls_pick].items():
        nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
        if len(nums) > 1 and gs.best_params_[k_] in (min(nums), max(nums)):
            print(f'!! {k_}={gs.best_params_[k_]} อยู่ขอบ grid -> ลองขยาย grid')
else:
    clf_final = pipe.fit(X_train, yc_train)
    print('โมเดล:', cls_pick, '(ไม่จูน)')

### 3.3 ผล Stage 1 บน test (accuracy / F1 / confusion matrix)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred_c = clf_final.predict(X_test)
print(f'Accuracy     = {accuracy_score(yc_test, pred_c):.4f}')
print(f'F1 macro     = {f1_score(yc_test, pred_c, average="macro"):.4f}')
print(f'F1 weighted  = {f1_score(yc_test, pred_c, average="weighted"):.4f}')
if K == 2:
    print(f'F1 (positive = "{class_names[1]}") = {f1_score(yc_test, pred_c):.4f}')
print()
print(classification_report(yc_test, pred_c, target_names=class_names, digits=3))

fig, ax = plt.subplots(figsize=(4.5 + 0.4 * K, 4 + 0.3 * K))
ConfusionMatrixDisplay(confusion_matrix(yc_test, pred_c), display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion matrix (test)'); plt.show()

## 4. สร้าง feature ให้ Stage 2 (ส่วนสำคัญที่สุดของ pipeline นี้)

ถ้าเอา class ที่ `clf_final` ทำนายบน **train** ไปเป็น feature ตรงๆ มันจะแม่นเกินจริง (model เคยเห็นแถวเหล่านั้นแล้ว) แล้ว Stage 2 จะเชื่อ feature นั้นมากเกินไป พอถึง test ที่ class แม่นน้อยลง R² จะตก

วิธีแก้: ใช้ **out-of-fold prediction** บน train (`cross_val_predict`) คือแต่ละแถวถูกทำนายโดยโมเดลที่ไม่เคยเห็นแถวนั้น ส่วน test ใช้ `clf_final` ทำนายตามปกติ

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

# features หลัง preprocess (fit จาก train เท่านั้น)
prep_final = clf_final.named_steps['prep']
Xtr_p = prep_final.transform(X_train)
Xte_p = prep_final.transform(X_test)
base_names = list(prep_final.get_feature_names_out())

# probability: train = out-of-fold, test = จากโมเดลสุดท้าย
oof_proba = cross_val_predict(clone(clf_final), X_train, yc_train, cv=cv_cls, method='predict_proba')
te_proba  = clf_final.predict_proba(X_test)

print(f'accuracy (OOF บน train) = {accuracy_score(yc_train, oof_proba.argmax(1)):.4f}')
print(f'accuracy (test)         = {accuracy_score(yc_test, te_proba.argmax(1)):.4f}   <- สองค่านี้ควรใกล้กัน')
print(f'accuracy (train แบบตรงๆ ไม่ใช่ OOF) = {accuracy_score(yc_train, clf_final.predict(X_train)):.4f}   <- สูงเกินจริง ห้ามใช้เป็น feature')

def make_stage2(Xp, proba):
    parts = [Xp]
    if ADD_PROBA:
        parts.append(proba)
    if ADD_CLASS_ONEHOT:
        parts.append(np.eye(K)[proba.argmax(1)])
    return np.hstack(parts)

z_names = list(base_names)
if ADD_PROBA:        z_names += [f'proba_{c}' for c in class_names]
if ADD_CLASS_ONEHOT: z_names += [f'predclass_{c}' for c in class_names]

Ztr, Zte = make_stage2(Xtr_p, oof_proba), make_stage2(Xte_p, te_proba)                         # ใช้งานจริง
Ztr_true, Zte_true = make_stage2(Xtr_p, np.eye(K)[yc_train]), make_stage2(Xte_p, np.eye(K)[yc_test])   # เฉลย class (เพดานบน)
print('Stage-2 feature shape:', Ztr.shape, Zte.shape)

## 5. Stage 2: Regression (ทำนาย y จาก feature เดิม + class ที่ทำนาย)

### 5.1 เทียบโมเดล (จัดอันดับด้วย CV R² บน train)

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

if USE_LOG_TARGET and yr_train.min() <= -1:
    print('y มีค่า <= -1 ใช้ log1p ไม่ได้ -> ปิด USE_LOG_TARGET')
    USE_LOG_TARGET = False

def wrap_log(m):
    return TransformedTargetRegressor(regressor=m, func=np.log1p, inverse_func=np.expm1) if USE_LOG_TARGET else m

REG_MODELS = {
    'Baseline(mean)': lambda: DummyRegressor(),
    'Linear':       lambda: wrap_log(LinearRegression()),
    'Ridge':        lambda: wrap_log(Ridge(alpha=1.0)),
    'Lasso':        lambda: wrap_log(Lasso(alpha=0.1, max_iter=20000)),
    'ElasticNet':   lambda: wrap_log(ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000)),
    'KNN':          lambda: wrap_log(KNeighborsRegressor(n_neighbors=7)),
    'SVR':          lambda: TransformedTargetRegressor(regressor=SVR(C=10), transformer=StandardScaler()),   # SVR ต้อง scale y ด้วย
    'DecisionTree': lambda: wrap_log(DecisionTreeRegressor(max_depth=6, random_state=SEED)),
    'RandomForest': lambda: wrap_log(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)),
    'GradBoost':    lambda: wrap_log(GradientBoostingRegressor(random_state=SEED)),
}
cv_reg = KFold(N_SPLITS, shuffle=True, random_state=SEED)

def rank_regressors(Ztr_, Zte_):
    rows = []
    for name, mk in REG_MODELS.items():
        m = mk()
        cvs = cross_val_score(m, Ztr_, yr_train, cv=cv_reg, scoring='r2', n_jobs=-1)
        m.fit(Ztr_, yr_train)
        p = m.predict(Zte_)
        rows.append({'model': name, 'R2_train': r2_score(yr_train, m.predict(Ztr_)),
                     'CV_R2': cvs.mean(), 'CV_R2_std': cvs.std(), 'R2_test': r2_score(yr_test, p),
                     'RMSE_test': np.sqrt(mean_squared_error(yr_test, p)), 'MAE_test': mean_absolute_error(yr_test, p)})
    out = pd.DataFrame(rows).sort_values('CV_R2', ascending=False).reset_index(drop=True)
    out.insert(0, 'rank', out.index + 1)
    return out

reg_df = rank_regressors(Ztr, Zte)
display(reg_df.round(4))
print('อ่านผล: R2_train สูงมากแต่ CV/test ต่ำ = overfit | ต้องชนะ Baseline(mean) ซึ่งได้ R2 ≈ 0')

### 5.2 Ablation: Stage 1 ช่วย Stage 2 จริงไหม?
เทียบโมเดล regression ตัวเดียวกัน 3 แบบ
- **ไม่ใช้ class** = ไม่มี Stage 1 เลย
- **ใช้ class ที่ทำนาย (OOF)** = flow จริงของโจทย์
- **ใช้ class จริง** = เพดานบน (ถ้า Stage 1 แม่น 100%) *ใช้เปรียบเทียบเท่านั้น ห้ามใช้ตอนส่งผล*

ตีความ: "ทำนาย" ≈ "ไม่ใช้" → class ไม่ช่วย | "ทำนาย" << "จริง" → ถ้า Stage 1 แม่นขึ้นจะช่วยมาก (ลงแรงไปที่ Stage 1)

In [ ]:
best_reg_name = reg_df[reg_df.model != 'Baseline(mean)'].iloc[0]['model']
print('โมเดล regression ที่ใช้เทียบ:', best_reg_name, '\n')

sets = {'ไม่ใช้ class':              (Xtr_p, Xte_p),
        'ใช้ class ที่ทำนาย (OOF)':  (Ztr, Zte),
        'ใช้ class จริง (เพดาน)':    (Ztr_true, Zte_true)}
rows = []
for label, (A, B) in sets.items():
    m = REG_MODELS[best_reg_name]()
    cvs = cross_val_score(m, A, yr_train, cv=cv_reg, scoring='r2', n_jobs=-1)
    m.fit(A, yr_train); p = m.predict(B)
    rows.append({'feature set': label, 'CV_R2': cvs.mean(), 'R2_test': r2_score(yr_test, p),
                 'RMSE_test': np.sqrt(mean_squared_error(yr_test, p)), 'MAE_test': mean_absolute_error(yr_test, p)})
display(pd.DataFrame(rows).round(4))

### 5.3 จูน hyperparameter ของโมเดลที่ชนะ + fit ตัวสุดท้าย (`reg_final`)

In [ ]:
TUNE_REG = True
REG_PICK = None        # None = อันดับ 1 (ไม่นับ baseline) | หรือระบุชื่อ เช่น 'GradBoost'

GRIDS_REG = {
    'Ridge':        {'alpha': [0.01, 0.1, 1, 10, 100, 1000]},
    'Lasso':        {'alpha': [0.001, 0.01, 0.1, 1, 10]},
    'ElasticNet':   {'alpha': [0.01, 0.1, 1], 'l1_ratio': [0.2, 0.5, 0.8]},
    'KNN':          {'n_neighbors': [3, 5, 10, 15, 25], 'weights': ['uniform', 'distance']},
    'SVR':          {'C': [1, 10, 100], 'epsilon': [0.01, 0.1, 0.3], 'gamma': ['scale', 0.01]},
    'DecisionTree': {'max_depth': [3, 5, 8, 12], 'min_samples_leaf': [1, 5, 10, 20]},
    'RandomForest': {'max_depth': [None, 6, 10, 16], 'min_samples_leaf': [1, 3, 5], 'max_features': [1.0, 0.5]},
    'GradBoost':    {'n_estimators': [100, 200, 300], 'learning_rate': [0.03, 0.1], 'max_depth': [2, 3, 4]},
}

reg_pick = REG_PICK or reg_df[reg_df.model != 'Baseline(mean)'].iloc[0]['model']
base_model = REG_MODELS[reg_pick]()
if TUNE_REG and reg_pick in GRIDS_REG:
    pre = 'regressor__' if isinstance(base_model, TransformedTargetRegressor) else ''
    grid = {pre + k: v for k, v in GRIDS_REG[reg_pick].items()}
    gs = GridSearchCV(base_model, grid, cv=cv_reg, scoring='r2', n_jobs=-1)
    gs.fit(Ztr, yr_train)
    reg_final = gs.best_estimator_
    print('โมเดล:', reg_pick, '| best params:', gs.best_params_, f'| CV R2={gs.best_score_:.4f}')
    for k_, vals in grid.items():
        nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
        if len(nums) > 1 and gs.best_params_[k_] in (min(nums), max(nums)):
            print(f'!! {k_}={gs.best_params_[k_]} อยู่ขอบ grid -> ลองขยาย grid')
else:
    reg_final = base_model.fit(Ztr, yr_train)
    print('โมเดล:', reg_pick, '(ไม่จูน)')

p_reg = reg_final.predict(Zte)
print(f'\nStage 2 บน test:  R2={r2_score(yr_test, p_reg):.4f}  RMSE={np.sqrt(mean_squared_error(yr_test, p_reg)):.4f}  MAE={mean_absolute_error(yr_test, p_reg):.4f}')
print(f'R2 train = {r2_score(yr_train, reg_final.predict(Ztr)):.4f}  (เทียบกับ test เช็ค overfit)')

## 6. ทำนายแบบ end-to-end (เหมือนตอนสอบที่ได้ test set มา)

`predict_all(df_new)` รับ dataframe ดิบ (ไม่ต้องมีคอลัมน์ class/y) → ทำ Stage 1 → สร้าง feature → Stage 2 → คืนทั้ง class และค่า y

In [ ]:
import numpy as np, pandas as pd

def predict_all(df_new):
    Xn = clean_X(df_new)
    proba = clf_final.predict_proba(Xn)
    Zn = make_stage2(clf_final.named_steps['prep'].transform(Xn), proba)
    out = pd.DataFrame({'pred_class': le.inverse_transform(proba.argmax(1)),
                        'pred_value': reg_final.predict(Zn)}, index=df_new.index)
    for i, c in enumerate(class_names):
        out['proba_' + c] = proba[:, i]
    return out

# --- ทดสอบกับ test set (ส่ง "แถวดิบ" เข้าไป เหมือนข้อมูลใหม่) ---
test_raw = data.iloc[idx_te]
res = predict_all(test_raw)

print('=== ผลรวม end-to-end บน test ===')
print(f'[Stage 1] Accuracy={accuracy_score(test_raw[CLASS_COL], res.pred_class):.4f}  '
      f'F1 macro={f1_score(test_raw[CLASS_COL], res.pred_class, average="macro"):.4f}  '
      f'F1 weighted={f1_score(test_raw[CLASS_COL], res.pred_class, average="weighted"):.4f}')
print(f'[Stage 2] R2={r2_score(yr_test, res.pred_value):.4f}  '
      f'RMSE={np.sqrt(mean_squared_error(yr_test, res.pred_value)):.4f}  MAE={mean_absolute_error(yr_test, res.pred_value):.4f}')
display(res.head(10).round(3))

# ถ้าต้องส่งไฟล์ผลทำนาย:
# res[['pred_class', 'pred_value']].to_csv('predictions.csv', index=False)
# และถ้ามี test set ของอาจารย์แยกมา: predict_all(test_df)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, name in enumerate(class_names):
    mk = yc_test == k
    ax[0].scatter(yr_test[mk], res.pred_value.to_numpy()[mk], alpha=0.6, label=name)
lo, hi = min(yr_test.min(), res.pred_value.min()), max(yr_test.max(), res.pred_value.max())
ax[0].plot([lo, hi], [lo, hi], 'r--'); ax[0].legend(title='class จริง')
ax[0].set_xlabel('Actual'); ax[0].set_ylabel('Predicted'); ax[0].set_title('Actual vs Predicted (สีตาม class จริง)')
ax[1].scatter(res.pred_value, yr_test - res.pred_value, alpha=0.5); ax[1].axhline(0, color='r', linestyle='--')
ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Residual'); ax[1].set_title('Residual plot')
plt.tight_layout(); plt.show()

## 7. ส่วนเสริม (เลือกรันตามต้องการ)

### 7.1 ทางเลือก: แยกโมเดล regression ต่อ class
แทนที่จะส่ง class เป็น feature ให้ฝึกโมเดล regression แยกของแต่ละ class แล้วเลือกใช้ตาม class ที่ทำนาย เหมาะเมื่อ y ของแต่ละ class มีรูปแบบต่างกันมาก (ดู eta² / boxplot ในส่วน 1) แต่แต่ละ class ต้องมีข้อมูลพอ

In [ ]:
MIN_PER_CLASS = 30     # class ที่มีข้อมูลน้อยกว่านี้ใช้โมเดลรวมแทน

glob = clone(reg_final).fit(Xtr_p, yr_train)                    # โมเดลรวม (ไม่มี feature class)
per = {}
for k in range(K):
    mk = yc_train == k
    if mk.sum() >= MIN_PER_CLASS:
        per[k] = clone(reg_final).fit(Xtr_p[mk], yr_train[mk])
print('class ที่มีโมเดลของตัวเอง:', [class_names[k] for k in per])

te_cls = te_proba.argmax(1)
p_pc = glob.predict(Xte_p)
for k, m_ in per.items():
    mk = te_cls == k
    if mk.any():
        p_pc[mk] = m_.predict(Xte_p[mk])

print(f'Per-class regression : R2={r2_score(yr_test, p_pc):.4f}  RMSE={np.sqrt(mean_squared_error(yr_test, p_pc)):.4f}  MAE={mean_absolute_error(yr_test, p_pc):.4f}')
print(f'Class-as-feature     : R2={r2_score(yr_test, p_reg):.4f}  RMSE={np.sqrt(mean_squared_error(yr_test, p_reg)):.4f}  MAE={mean_absolute_error(yr_test, p_reg):.4f}')

### 7.2 อะไรสำคัญต่อการทำนาย (permutation importance)
ดูด้วยว่า feature ที่มาจาก class (`proba_*`, `predclass_*`) ถูกใช้จริงแค่ไหน

In [ ]:
from sklearn.inspection import permutation_importance

# Stage 2: feature ทั้งหมด (รวม class)
r2_ = permutation_importance(reg_final, Zte, yr_test, n_repeats=10, random_state=SEED, scoring='r2')
imp2 = pd.Series(r2_.importances_mean, index=z_names).sort_values(ascending=False)
print('Stage 2 (regression) top 12:'); print(imp2.head(12).round(4))

# Stage 1: คอลัมน์ดิบ (อ่านง่าย)
r1_ = permutation_importance(clf_final, X_test, yc_test, n_repeats=10, random_state=SEED, scoring=CLS_METRIC)
imp1 = pd.Series(r1_.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(f'\nStage 1 (classification, drop ใน {CLS_METRIC}) top 12:'); print(imp1.head(12).round(4))

---
## สรุปแนวทางตอนสอบ

1. รัน 0 → 1 → 2 (แก้ `CONFIG` ก่อน) → 3 → 4 → 5 → 6 ได้ผลครบ
2. เล่า flow ให้อาจารย์: stratified split → เทียบหลาย classifier ด้วย CV (F1/accuracy) → จูน → ใช้ **OOF prediction** สร้าง feature ให้ Stage 2 (กัน leakage) → เทียบหลาย regressor ด้วย CV R² → จูน → ทำนาย end-to-end บน test
3. ถ้า class ไม่สมดุล: เน้น F1 macro, ลอง `CLASS_WEIGHT = 'balanced'`
4. ถ้า y เบ้: `USE_LOG_TARGET = True` แล้วเทียบ R² ดู
5. ถ้าอยากให้ Stage 1 "ทายตรง" ขึ้นอีก ลองเพิ่ม feature engineering, ปรับ `CLS_METRIC`, หรือเพิ่ม grid ของโมเดลที่ชนะ
6. ดูตาราง ablation (5.2) เพื่อตอบคำถามว่า "Stage 1 ช่วย Stage 2 ไหม" ซึ่งอาจารย์อาจถาม